# AWS Marketplace Simulator — End-to-End Walkthrough

This notebook demonstrates the complete subscription lifecycle against the **simulator**:

1. **Seller**: create a product with contract + pay-as-you-go pricing, define dimensions, enable a free trial, register a lifecycle webhook.
2. **Seller**: issue a private offer to a specific buyer account.
3. **Buyer**: accept the offer (simulator POSTs a `x-amzn-marketplace-token` to the fulfillment URL).
4. **Seller production code** (plain `boto3`): `ResolveCustomer` → `GetEntitlements` → `BatchMeterUsage`.
5. **Buyer**: unsubscribe; entitlement calls go empty, metering flagged `CustomerNotSubscribed`.

Everything the seller's **production** code does uses unmodified `boto3` clients pointing at the simulator via `AWS_ENDPOINT_URL_*`. To migrate to real AWS Marketplace, unset those env vars.

---

## 0. Start the simulator in a background thread

For a real deployment you'd run `python -m mp_simulator serve --port 9999 --db mp-sim.sqlite` in its own process. For the notebook, we start it inline.

In [1]:
import json
import os
import sys
import tempfile
import time

sys.path.insert(0, "..")

from mp_simulator import server, clock

clock.reset()

db_file = tempfile.NamedTemporaryFile(prefix="mp-sim-demo-", suffix=".sqlite", delete=False).name
srv, port = server.serve_in_thread(host="127.0.0.1", port=0, db_path=db_file)
BASE_URL = f"http://127.0.0.1:{port}"
print("simulator listening at", BASE_URL)

os.environ["AWS_ENDPOINT_URL_MARKETPLACE_METERING"] = BASE_URL
os.environ["AWS_ENDPOINT_URL_MARKETPLACE_ENTITLEMENT_SERVICE"] = BASE_URL
os.environ.setdefault("AWS_ACCESS_KEY_ID", "test")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "test")
os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")

simulator listening at http://127.0.0.1:45355


'us-east-1'

## 1. Seller admin: create a product

Uses the `MpSimulatorClient` helper for the admin REST surface (the simulator's stand-in for AMMP). The contract + overage dimensions map directly to how we'd configure the real Marketplace listing.


In [2]:
from client.mp_simulator_client import MpSimulatorClient

admin = MpSimulatorClient(BASE_URL)

product = admin.create_product(
    name="GenAI IDP Accelerator — AutoTune (test listing)",
    pricingModel="contract-with-payg",
    trialDays=30,
    fulfillmentUrl=None,  # no webhook for the notebook demo
    quickLaunchTemplateUrl="s3://idp-mp-harness-artifacts/test-feature/test-feature.yaml",
    dimensions=[
        {
            "apiName": "cap_docs",
            "displayName": "Capacity (docs/mo)",
            "category": "Units",
            "unitPrice": 0.05,
            "kind": "contract",
        },
        {
            "apiName": "docs_used",
            "displayName": "Documents processed",
            "category": "Units",
            "unitPrice": 0.0,
            "kind": "usage",
        },
        {
            "apiName": "docs_over",
            "displayName": "Overage docs",
            "category": "Units",
            "unitPrice": 0.002,
            "kind": "overage",
        },
    ],
)
print("product_code:", product["product_code"])
print(json.dumps(product, indent=2, default=str))

product_code: mp-sim-87e0ac3ee569
{
  "product_code": "mp-sim-87e0ac3ee569",
  "license_arn": "arn:aws:license-manager::sim:license/mp-sim-87e0ac3ee569",
  "name": "GenAI IDP Accelerator \u2014 AutoTune (test listing)",
  "pricing_model": "contract-with-payg",
  "published": 0,
  "trial_days": 30,
  "fulfillment_url": null,
  "quick_launch_template_url": "s3://idp-mp-harness-artifacts/test-feature/test-feature.yaml",
  "dimensions_json": "[{\"apiName\": \"cap_docs\", \"displayName\": \"Capacity (docs/mo)\", \"category\": \"Units\", \"unitPrice\": 0.05, \"kind\": \"contract\"}, {\"apiName\": \"docs_used\", \"displayName\": \"Documents processed\", \"category\": \"Units\", \"unitPrice\": 0.0, \"kind\": \"usage\"}, {\"apiName\": \"docs_over\", \"displayName\": \"Overage docs\", \"category\": \"Units\", \"unitPrice\": 0.002, \"kind\": \"overage\"}]",
  "created_at": 1777987904.0421999,
  "updated_at": 1777987904.0421999,
  "dimensions": [
    {
      "apiName": "cap_docs",
      "displayNa

## 2. Seller admin: register a lifecycle sink and issue a private offer

In production the lifecycle sink would be an SNS topic that fans out to SQS → Lambda. Here we use an **in-process callback** for visibility — the simulator also supports `webhook` (any HTTP URL) and `sns` (real AWS SNS publish).

In [3]:
from mp_simulator import notifications as mp_notifications

captured_events = []


def capture(envelope):
    captured_events.append(envelope)


mp_notifications.register_inproc("demo-sink", capture)

admin.create_lifecycle_sink(
    productCode=product["product_code"],
    transport="inproc",
    target="demo-sink",
    topic="subscription",
)
admin.create_lifecycle_sink(
    productCode=product["product_code"], transport="inproc", target="demo-sink", topic="entitlement"
)

offer = admin.create_offer(
    productCode=product["product_code"],
    kind="private",
    buyerAccountAllowlist=["123456789012"],
    contractTier={"dimension": "cap_docs", "quantity": 500},
    durationMonths=1,
    freeTrialEnabled=True,
)
print("offer_id:", offer["offer_id"], "(private, 500 docs/mo, 30-day trial)")

offer_id: offer-1e6648540c (private, 500 docs/mo, 30-day trial)


## 3. Buyer: accept the offer

Simulator does everything real Marketplace would:
- checks the allowlist (would reject `999999999999`)
- creates a subscription in `trial` status with `trialEndsAt = now + 30 days`
- mints a 1-hour-lived `x-amzn-marketplace-token`
- POSTs it to the product's fulfillment URL if one is configured
- emits `subscribe-success` on the subscription topic
- seeds the buyer's entitlement from the contract tier


In [4]:
buyer = MpSimulatorClient(BASE_URL)
sub = buyer.subscribe(offerId=offer["offer_id"], buyerAccountId="123456789012")
print(json.dumps(sub, indent=2, default=str))

{
  "customerIdentifier": "cust-0fe83069481f",
  "customerAWSAccountId": "123456789012",
  "productCode": "mp-sim-87e0ac3ee569",
  "offerId": "offer-1e6648540c",
  "status": "trial",
  "trialEndsAt": 1780579904.0881271,
  "registrationToken": "mp-sim-tok-T8XXyDA3UCNqlaajavj8Qw",
  "fulfillmentPostStatus": "skipped",
  "agreementId": "agmt-aa17226dca3c4c81"
}


**Lifecycle event the seller's webhook/SQS consumer would receive:**

In [5]:
print(json.dumps(json.loads(captured_events[0]["Message"]), indent=2, default=str))

{
  "action": "subscribe-success",
  "customer-identifier": "cust-0fe83069481f",
  "product-code": "mp-sim-87e0ac3ee569",
  "offer-identifier": "offer-1e6648540c",
  "isFreeTrialTermPresent": "true",
  "message-time": 1777987904.0885057
}


## 4. Seller production code — plain boto3, no mocks

**This is the critical section.** Every call below is unmodified boto3 SDK code.
The ONLY thing making it hit the simulator instead of real AWS is the two
`AWS_ENDPOINT_URL_*` env vars we set at the top of the notebook.

In [6]:
import boto3

mp = boto3.client("meteringmarketplace", region_name="us-east-1")
ent = boto3.client("marketplace-entitlement", region_name="us-east-1")

# 4a. ResolveCustomer: exchange the token we got on the fulfillment POST
resolved = mp.resolve_customer(RegistrationToken=sub["registrationToken"])
print("ResolveCustomer ->", json.dumps(resolved, indent=2, default=str))

ResolveCustomer -> {
  "CustomerIdentifier": "cust-0fe83069481f",
  "ProductCode": "mp-sim-87e0ac3ee569",
  "CustomerAWSAccountId": "123456789012",
  "ResponseMetadata": {
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "server": "mp-sim/0.1 Python/3.12.12",
      "date": "Tue, 05 May 2026 13:31:44 GMT",
      "content-type": "application/x-amz-json-1.1",
      "content-length": "121"
    },
    "RetryAttempts": 0
  }
}


In [7]:
# 4b. GetEntitlements: the seller's 'validate' API path
result = ent.get_entitlements(
    ProductCode=product["product_code"],
    Filter={"CUSTOMER_IDENTIFIER": [resolved["CustomerIdentifier"]]},
)
print(json.dumps(result, indent=2, default=str))

{
  "Entitlements": [
    {
      "ProductCode": "mp-sim-87e0ac3ee569",
      "Dimension": "cap_docs",
      "CustomerIdentifier": "cust-0fe83069481f",
      "Value": {
        "IntegerValue": 500
      },
      "ExpirationDate": "2026-06-04 13:31:44.088194+00:00"
    }
  ],
  "ResponseMetadata": {
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "server": "mp-sim/0.1 Python/3.12.12",
      "date": "Tue, 05 May 2026 13:31:44 GMT",
      "content-type": "application/x-amz-json-1.1",
      "content-length": "195"
    },
    "RetryAttempts": 0
  }
}


In [8]:
# 4c. BatchMeterUsage: the seller's hourly roll-up
resp = mp.batch_meter_usage(
    ProductCode=product["product_code"],
    UsageRecords=[
        {
            "Timestamp": int(time.time()),
            "CustomerIdentifier": resolved["CustomerIdentifier"],
            "Dimension": "docs_used",
            "Quantity": 42,
        }
    ],
)
print(json.dumps(resp, indent=2, default=str))

{
  "Results": [
    {
      "UsageRecord": {
        "Timestamp": "2026-05-05 13:31:44+00:00",
        "CustomerIdentifier": "cust-0fe83069481f",
        "Dimension": "docs_used",
        "Quantity": 42
      },
      "MeteringRecordId": "mri-4d68fb19-c8ba-45e4-9f51-ebe1a9a70fc4",
      "Status": "Success"
    }
  ],
  "UnprocessedRecords": [],
  "ResponseMetadata": {
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "server": "mp-sim/0.1 Python/3.12.12",
      "date": "Tue, 05 May 2026 13:31:44 GMT",
      "content-type": "application/x-amz-json-1.1",
      "content-length": "255"
    },
    "RetryAttempts": 0
  }
}


**Inspecting what landed** — the admin API lets us peek at the simulator's state (useful in tests, not available in real AWS).

In [9]:
print("usage log:")
for u in admin.list_usage(product_code=product["product_code"]):
    print(" ", u["dimension"], "qty=", u["quantity"], "status=", u["status"])

usage log:
  docs_used qty= 42 status= Success


## 5. Buyer unsubscribes

Simulator emits `unsubscribe-pending`, cancels the subscription (real AWS gives the seller ~1 hour after `pending`), deletes the entitlement rows, then emits `unsubscribe-success` and `entitlement-updated`.

In [10]:
buyer.unsubscribe(customerIdentifier=resolved["CustomerIdentifier"])

# GetEntitlements now returns an empty set
r = ent.get_entitlements(
    ProductCode=product["product_code"],
    Filter={"CUSTOMER_IDENTIFIER": [resolved["CustomerIdentifier"]]},
)
print("Entitlements after cancel:", r["Entitlements"])

# BatchMeterUsage flags the record
r = mp.batch_meter_usage(
    ProductCode=product["product_code"],
    UsageRecords=[
        {
            "Timestamp": int(time.time()),
            "CustomerIdentifier": resolved["CustomerIdentifier"],
            "Dimension": "docs_used",
            "Quantity": 1,
        }
    ],
)
print("Meter status after cancel:", r["Results"][0]["Status"])

Entitlements after cancel: []
Meter status after cancel: CustomerNotSubscribed


**Full lifecycle event sequence** (what the seller's SNS/SQS consumer would have seen end-to-end):

In [11]:
for env in captured_events:
    msg = json.loads(env["Message"])
    print(
        f"{msg['action']:<25}  customer={msg['customer-identifier']}  product={msg['product-code']}"
    )

subscribe-success          customer=cust-0fe83069481f  product=mp-sim-87e0ac3ee569
unsubscribe-pending        customer=cust-0fe83069481f  product=mp-sim-87e0ac3ee569
unsubscribe-success        customer=cust-0fe83069481f  product=mp-sim-87e0ac3ee569
entitlement-updated        customer=cust-0fe83069481f  product=mp-sim-87e0ac3ee569


## 6. Time-travel: test the 30-day trial expiry

Real tests would need to wait 30 days. The simulator's `advance_time` bumps its internal clock so entitlements time out and trial deadlines pass instantly.

In [12]:
# The offer in cell 2 was private (only 123456789012 allowlisted). For the
# trial-expiry demo we use a brand-new public offer so a different buyer account
# can subscribe and get a trial.
public_offer = admin.create_offer(
    productCode=product["product_code"],
    kind="public",
    contractTier={"dimension": "cap_docs", "quantity": 500},
    durationMonths=1,
    freeTrialEnabled=True,
)
trial_sub = buyer.subscribe(offerId=public_offer["offer_id"], buyerAccountId="555555555555")
print("trialEndsAt:", trial_sub["trialEndsAt"])

# 25 days in: still entitled
admin.advance_time(25 * 86400)
r = ent.get_entitlements(
    ProductCode=product["product_code"],
    Filter={"CUSTOMER_IDENTIFIER": [trial_sub["customerIdentifier"]]},
)
print("day 25 entitlements:", len(r["Entitlements"]))

# 35 days in: entitlement expired (based on contract duration)
admin.advance_time(10 * 86400)
r = ent.get_entitlements(
    ProductCode=product["product_code"],
    Filter={"CUSTOMER_IDENTIFIER": [trial_sub["customerIdentifier"]]},
)
print("day 35 entitlements:", len(r["Entitlements"]))

trialEndsAt: 1780579904.323494
day 25 entitlements: 1
day 35 entitlements: 0


## 7. Shutdown

In [13]:
srv.shutdown()
os.unlink(db_file)
print("simulator stopped")

simulator stopped


---
## What we just proved

| Capability | Step |
|---|---|
| Catalog CRUD (stand-in for AMMP) | 1–2 |
| Private offer issuance + allowlist | 2 |
| Subscribe flow with token minting + fulfillment URL POST | 3 |
| Lifecycle event delivery (matches SNS envelope shape) | 3, 5 |
| **Plain `boto3.client('meteringmarketplace')` works unchanged** | 4a–c |
| **Plain `boto3.client('marketplace-entitlement')` works unchanged** | 4b |
| Unsubscribe → entitlement empty, metering flagged | 5 |
| Time advancement for trial / expiry tests | 6 |

### Migrating to real AWS Marketplace

Unset the endpoint-url env vars and point your seller backend at the real AWS service endpoints. **No boto3 call site changes.** The only thing that changes is the admin/buyer pieces (steps 1–3) because those replicate the AMMP UI / the buyer console — in prod, a Marketplace onboarding engineer creates the listing for you and real customers accept it through the AWS console.
